# 📈 Tutorial 2 (T2): Linear Regression & Regularization in Python
### Machine Learning for Precision Agriculture (Predicting Rainfall / Soil Moisture)

**Goal:** Learn how Linear Regression works to predict continuous values and explore Regularization techniques:
1. **Linear Regression (OLS)**: Fitting an optimal linear hyperplane ($y = w_1 X_1 + ... + w_n X_n + b$).
2. **Model Evaluation**: Calculating MAE, MSE, RMSE, and $R^2$ Score.
3. **Regularization (Ridge $L_2$ vs Lasso $L_1$)**: Preventing overfitting and balancing the Bias-Variance tradeoff.
4. **Visualizations**: Plotting Actual vs. Predicted values and Feature Coefficients.

--- 
## ⚙️ Step 0: Import Libraries & Setup Data (For Google Colab)

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Set plot styling
sns.set_theme(style="whitegrid")
plt.rcParams.update({'figure.dpi': 120})

# Ensure data directory exists
os.makedirs("data", exist_ok=True)
crop_csv = os.path.join("data", "Crop_recommendation.csv")

if not os.path.exists(crop_csv):
    print("Creating sample Crop_recommendation.csv for Google Colab...")
    crops = ["rice", "maize", "chickpea", "kidneybeans", "pigeonpeas", "mothbeans", "mungbean", "blackgram", "lentil", "pomegranate"]
    df_c = pd.DataFrame({
        'N': np.random.randint(10, 140, 200),
        'P': np.random.randint(5, 145, 200),
        'K': np.random.randint(15, 205, 200),
        'temperature': np.random.uniform(8.0, 43.0, 200),
        'humidity': np.random.uniform(14.0, 100.0, 200),
        'ph': np.random.uniform(3.5, 9.9, 200),
        'rainfall': np.random.uniform(20.0, 300.0, 200),
        'label': np.random.choice(crops, 200)
    })
    df_c.to_csv(crop_csv, index=False)

print("[OK] Setup complete! Data is ready.")

--- 
## 🎯 Step 1: Define Target & Feature Variables
We use soil nutrients ($N, P, K$) and climate features (temperature, humidity, pH) to predict continuous **Rainfall** requirement.

In [ ]:
df = pd.read_csv(crop_csv)

feature_cols = ['N', 'P', 'K', 'temperature', 'humidity', 'ph']
target_col = 'rainfall'

X = df[feature_cols]
y = df[target_col]

print(f"Target Variable (Continuous): {target_col}")
print(f"Feature Variables: {feature_cols}")
display(df[feature_cols + [target_col]].head())

--- 
## ✂️ Step 2: Train-Test Split & Feature Scaling

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set shape: {X_train_scaled.shape}")
print(f"Testing set shape:  {X_test_scaled.shape}")

--- 
## 🤖 Step 3: Train Linear Regression, Ridge ($L_2$), and Lasso ($L_1$)
We evaluate OLS Linear Regression alongside **Ridge ($L_2$)** and **Lasso ($L_1$)** regularization.

In [ ]:
models = {
    "Linear Regression (OLS)": LinearRegression(),
    "Ridge Regression (L2)": Ridge(alpha=1.0),
    "Lasso Regression (L1)": Lasso(alpha=0.1)
}

results = []
predictions = {}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    predictions[name] = y_pred
    
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)
    
    results.append({
        'Model': name,
        'MAE': mae,
        'MSE': mse,
        'RMSE': rmse,
        'R2 Score': r2
    })

results_df = pd.DataFrame(results)
display(results_df)

--- 
## 📊 Step 4: Visualizations (Actual vs Predicted & Coefficients)

In [ ]:
# 4a. Actual vs Predicted Plot
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(y_test, predictions["Linear Regression (OLS)"], color='royalblue', alpha=0.7, label='Predicted Points')
ax.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Ideal Line (y = x)')
ax.set_title("Linear Regression: Actual vs Predicted Rainfall")
ax.set_xlabel("Actual Rainfall (mm)")
ax.set_ylabel("Predicted Rainfall (mm)")
ax.legend()
plt.tight_layout()
plt.show()

# 4b. Regularization Coefficients Comparison
lr_model = models["Linear Regression (OLS)"]
ridge_model = models["Ridge Regression (L2)"]
lasso_model = models["Lasso Regression (L1)"]

coef_df = pd.DataFrame({
    'Feature': feature_cols,
    'Linear Regression': lr_model.coef_,
    'Ridge (L2)': ridge_model.coef_,
    'Lasso (L1)': lasso_model.coef_
})

fig, ax = plt.subplots(figsize=(10, 5))
coef_df.set_index('Feature').plot(kind='bar', ax=ax)
ax.set_title("Feature Coefficients: Linear vs Ridge vs Lasso")
ax.set_ylabel("Coefficient Weight")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

--- 
## 💡 What I Learned from Tutorial 2 (Summary for Viva / Notebook)
1. **Linear Regression**: Fits an equation $y = b_0 + b_1 X_1 + ... + b_n X_n$ to predict continuous targets.
2. **Metrics**: Evaluated performance using MAE, MSE, RMSE, and $R^2$ Score ($0$ to $1$).
3. **Regularization**:
   - **Ridge ($L_2$)** shrinks feature coefficients smoothly to prevent overfitting.
   - **Lasso ($L_1$)** can shrink uninformative coefficients to 0, performing automatic feature selection.